# Experiment 3: Ablation Studies (Optimized)

**Dataset:** Pascal VOC, 100 epochs per ablation.

| # | Ablation | What Changes |
|---|----------|--------------|
| A4 | ReLU6 vs SiLU | Activation |
| A6 | Resolution sweep | 224, 320, 416, 640 |
| A9 | Mosaic on vs off | Augmentation |

**Speedup:** Using `--pretrained` and `--compile` for faster iteration.

In [ ]:
!rm -rf /content/tinyYOLO 2>/dev/null
!git clone https://github.com/ShMazumder/tinyYOLO.git /content/tinyYOLO
%cd /content/tinyYOLO
!pip install -e . -q && pip install tqdm timm -q
import torch, os, json, glob
print(f'GPU: {torch.cuda.get_device_name(0)}')
!python -c "from ultralytics.data.utils import check_det_dataset; check_det_dataset('VOC.yaml')" 2>/dev/null

In [ ]:
EPOCHS = 100
BATCH = 128
SEED = 42
FLAGS = "--pretrained --compile"

In [ ]:
print('A4: ReLU6 (quantized)...')
get_ipython().system(f'python scripts/train.py --task det --variant quantized --data voc.yaml --imgsz 416 --epochs {EPOCHS} --batch {BATCH} --seed {SEED} --name ablation-a4-relu6 {FLAGS}')
print('\nA4: SiLU (standard)...')
get_ipython().system(f'python scripts/train.py --task det --variant standard --data voc.yaml --imgsz 416 --epochs {EPOCHS} --batch {BATCH} --seed {SEED} --name ablation-a4-silu {FLAGS}')
print('✅ A4 complete')

In [ ]:
for res in [224, 320, 416, 640]:
    print(f'\nA6: {res}x{res}...')
    get_ipython().system(f'python scripts/train.py --task det --variant quantized --data voc.yaml --imgsz {res} --epochs {EPOCHS} --batch {BATCH} --seed {SEED} --name ablation-a6-{res} {FLAGS}')
print('✅ A6 complete')

In [ ]:
os.environ['TINYYOLO_NO_MOSAIC'] = '1'
get_ipython().system(f'python scripts/train.py --task det --variant quantized --data voc.yaml --imgsz 416 --epochs {EPOCHS} --batch {BATCH} --seed {SEED} --name ablation-a9-no-mosaic {FLAGS}')
del os.environ['TINYYOLO_NO_MOSAIC']
print('✅ A9 complete')